In [1]:
from simple_slurm import Slurm

import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine: walk up from the
# working dir until we hit a repo marker, then put that dir on sys.path.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from metab_processing.metab_travlr_config import PROJECT_DATA_DIR, DATA_DIR as METAB_DATA_DIR
import time
from datetime import timedelta


In [2]:
outlog = f'{METAB_DATA_DIR}/harreman_logs/run_harr{str(time.strftime("%Y%m%d_%H%M%S"))}.log'

# slurm = Slurm(
#     account='fc_wagnerlabfca',
#     partition='savio4_gpu',
#     qos='a5k_gpu4_normal',
#     gres='gpu:A5000:1',
#     cpus_per_task=8,
#     ignore_pbs=True,
#     job_name='Harreman_Run',
#     output=outlog,
#     time=timedelta(hours=3),
# )

slurm = Slurm(
    account='fc_wagnerlabfca',
    partition='savio3_gpu',
    qos='a40_gpu3_normal',
    gres='gpu:A40:1',
    cpus_per_task=8,
    ignore_pbs=True,
    job_name='Harreman_Run',
    output=outlog,
    time=timedelta(hours=15),
)

python_path = '/global/home/users/fosterangus/.conda/envs/harreman/bin/python'

slurm.sbatch(python_path + ' /global/home/users/fosterangus/Projects/MetabTravLR/SpaceTravLR/metab_processing/Harreman/run_full_harr.py')

Submitted batch job 35798361



35798361

## UC_Xenium — group by `cell_type`

Same job body (`run_full_harr.py`), pointed at the `UC_Xenium` dir and told to use the
`cell_type` obs column instead of `Tier1/Tier2/Tier3`. One job processes every sample
folder under `UC_Xenium/`; each writes `easy_download/harreman_outputs/cell_type/` plus the
`metabolite_selection.yaml` that SpaceTravLR consumes.

In [ ]:
UC_DATA_DIR = f'{METAB_DATA_DIR}/UC_Xenium'
RUN_SCRIPT = _root / 'metab_processing' / 'Harreman' / 'run_full_harr.py'
python_path = '/global/home/users/fosterangus/.conda/envs/harreman/bin/python'

uc_outlog = f'{METAB_DATA_DIR}/harreman_logs/run_harr_UC_{time.strftime("%Y%m%d_%H%M%S")}.log'
Path(uc_outlog).parent.mkdir(parents=True, exist_ok=True)   # sbatch opens --output before the job runs

uc_slurm = Slurm(
    account='fc_wagnerlabfca',
    partition='savio3_gpu',
    qos='a40_gpu3_normal',
    gres='gpu:A40:1',
    cpus_per_task=8,
    ignore_pbs=True,
    job_name='Harreman_UC',
    output=uc_outlog,
    time=timedelta(hours=15),
)

uc_slurm.sbatch(f'{python_path} {RUN_SCRIPT} --data-dir {UC_DATA_DIR} --cell-type-cols cell_type')

## Alexi_UC_Spliced — group by coarse ICI-5K annotation

Same job body, pointed at the `Alexi_UC_Spliced` dir (4 spatial slices of patient
`13473_HS4_UC`, from `Preprocess/alexi_data.ipynb`) and told to use the
`25_06_11_ICI_5K_Coarse_annotations` obs column. One job processes every slice folder;
each writes `easy_download/harreman_outputs/25_06_11_ICI_5K_Coarse_annotations/` plus the
`metabolite_selection.yaml` that SpaceTravLR consumes.

In [2]:
ALEXI_DATA_DIR = f'{METAB_DATA_DIR}/Alexi_UC_Spliced'
ALEXI_ANNOT = '25_06_11_ICI_5K_Coarse_annotations'
RUN_SCRIPT = _root / 'metab_processing' / 'Harreman' / 'run_full_harr.py'
python_path = '/global/home/users/fosterangus/.conda/envs/harreman/bin/python'

alexi_outlog = f'{METAB_DATA_DIR}/harreman_logs/run_harr_Alexi_{time.strftime("%Y%m%d_%H%M%S")}.log'
Path(alexi_outlog).parent.mkdir(parents=True, exist_ok=True)   # sbatch opens --output before the job runs

alexi_slurm = Slurm(
    account='fc_wagnerlabfca',
    partition='savio3_gpu',
    qos='a40_gpu3_normal',
    gres='gpu:A40:1',
    cpus_per_task=8,
    ignore_pbs=True,
    job_name='Harreman_Alexi',
    output=alexi_outlog,
    time=timedelta(hours=15),
)

alexi_slurm.sbatch(f'{python_path} {RUN_SCRIPT} --data-dir {ALEXI_DATA_DIR} --cell-type-cols {ALEXI_ANNOT}')

Submitted batch job 38247336



38247336